In [32]:
%pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import numpy as np
import pandas as pd

trans_df = pd.read_csv("./datasets/input_0.csv")
print("SIZE:", trans_df.size)
trans_df.head(5)

SIZE: 1100000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/08 02:21,70,10042B930,149772,812F8AB80,247.14,Brazil Real,247.14,Brazil Real,Credit Card,0
1,2022/09/06 17:41,27,8037C8D60,17213,8039A84C0,1262.73,Yuan,1262.73,Yuan,Cheque,0
2,2022/09/02 00:14,70,10042B6A8,12503,801BF7490,52760.95,Euro,52760.95,Euro,Cash,0
3,2022/09/02 14:01,211,813B8D2D0,52860,814FF3EA0,12826.55,Swiss Franc,12826.55,Swiss Franc,ACH,0
4,2022/09/05 05:11,2,808B7B3B0,226259,809CE60E0,2026.07,Rupee,2026.07,Rupee,Credit Card,0


In [34]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/14 00:00]


In [35]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [36]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 36644


In [37]:
# Analyze accounts.
accounts_df = pd.read_csv("datasets/accounts_0.csv")
print("SIZE:", accounts_df.shape[0])

SIZE: 69581


In [38]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 19996


In [39]:
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size > 5)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 1625


In [40]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions.sort_values(by=["Account","Account.1"], ascending=True)

,From Bank,Account,To Bank,Account.1,Amount Paid
92241,70,10042B660,20,8002B8890,20.73
35972,70,10042B660,531,8002F3DB0,13.43
82477,70,10042B660,12,8003654F0,24.98
14339,70,10042B660,20,8003A71B0,33.69
70933,70,10042B660,20,8003A71B0,33.69
...,...,...,...,...,...
92267,117916,81C0B2730,117916,81C0B2730,3.03
25274,124126,81C0EEE50,124126,81C0EEE50,9.61
67655,32770,81C1205D0,49308,81C120850,26.05
91186,52931,81C122600,52931,81C122600,5.62


In [41]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]].drop_duplicates().sort_values(by="Account", ascending=True)

,From Bank,Account,Bank Name,Amount Paid
6783,70,10042B660,Willows Thrift,1.948379e+08
3253,13,800086730,Mexico Bank #10,2.100000e-01
1478,11,8000FA930,Savings Bank of Madison,1.592827e+07
14696,3201,8000FB870,Fieldstone Credit Union,4.088200e+02
4742,20,80015EFB0,First Bank of Danbury,2.522765e+08
...,...,...,...,...
43649,321447,81C037050,Baltech Community Bank,1.355030e+03
34855,32770,81C1205D0,National Bank of Helena,2.605000e+01
44241,376852,81C16A970,Dawn Federal Bank,1.215283e+04
41876,222559,81C172920,Savings Bank of Columbus,1.535650e+06


In [42]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]].sort_values(by=["Account", "Amount Paid"], ascending=True)

,From Bank,Account,Payment Format,Amount Paid
11888,70,10042B660,Cash,0.01
8381,70,10042B660,Cash,0.09
15030,70,10042B660,Credit Card,0.43
2434,70,10042B660,Cash,0.54
3759,70,10042B660,Cash,1.02
...,...,...,...,...
14860,362540,81BE6B060,Cheque,61.74
3745,56239,81BEE2E00,Credit Card,24.76
13363,231905,81BF09CD0,ACH,3927.98
14227,332439,81BFE0B30,ACH,222.08


In [43]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(accounts_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 1)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

,Bank,Account


In [44]:
#Conversion dates of period [2022-09-01, 2022-09-05] with base USD
#Bitcoin rates taken from investing.com
#Rest of currencies from api.frankfurter.dev
conversion_rates_records = np.rec.array([
           ('2022/09/01', 1.4644, 5.1805, 1.314 , 0.97999, 6.9   , 1.0002, 0.86272, 3.3535, 79.543, 139.34, 20.189, 60.367, 3.75, 1.,  19793.1),
           ('2022/09/02', 1.4691, 5.2035, 1.3141, 0.98175, 6.9035, 1.0011, 0.86468, 3.3755, 79.719, 140.11, 20.085, 60.427, 3.75, 1., 199999. ),
           ('2022/09/03', 1.4691, 5.2056, 1.3138, 0.98207, 6.9046, 1.0013, 0.86478, 3.3791, 79.75 , 140.17, 20.081, 60.471, 3.75, 1.,  19831.4),
           ('2022/09/04', 1.4695, 5.2082, 1.3139, 0.98219, 6.9047, 1.0013, 0.8649 , 3.3815, 79.754, 140.22, 20.084, 60.461, 3.75, 1.,  19952.7),
           ('2022/09/05', 1.4722, 5.1786, 1.3142, 0.98273, 6.9216, 1.0068, 0.86813, 3.4006, 79.816, 140.49, 20.018, 60.737, 3.75, 1.,  20126.1)],
          dtype=[ ('Date', 'O'), ('Australian Dollar', '<f8'), ('Brazil Real', '<f8'), ('Canadian Dollar', '<f8'), ('Swiss Franc', '<f8'), ('Yuan', '<f8'), ('Euro', '<f8'), ('UK Pound', '<f8'), ('Shekel', '<f8'), ('Rupee', '<f8'), ('Yen', '<f8'), ('Mexican Peso', '<f8'), ('Ruble', '<f8'), ('Saudi Riyal', '<f8'), ('US Dollar', '<f8'), ('Bitcoin', '<f8')])
conversion_rates_df = pd.DataFrame.from_records(conversion_rates_records)
conversion_rates_df = conversion_rates_df.set_index("Date")

In [45]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df.apply(lambda row: row['Amount Paid'] / conversion_rates_df[row['Payment Currency']][row["Timestamp"].split(" ")[0]], axis=1)
trans_sept_1st_wire_or_ach_filtered = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1.0]
print("SIZE:", trans_sept_1st_wire_or_ach_filtered.shape[0])

SIZE: 155


In [46]:
low_profile_transactions.info()

<class 'pandas.DataFrame'>
Index: 4881 entries, 9 to 99991
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   From Bank    4881 non-null   int64  
 1   Account      4881 non-null   str    
 2   To Bank      4881 non-null   int64  
 3   Account.1    4881 non-null   str    
 4   Amount Paid  4881 non-null   float64
dtypes: float64(1), int64(2), str(2)
memory usage: 228.8 KB


In [47]:
trans_usd_sept_2nd_df.info()

<class 'pandas.DataFrame'>
Index: 16648 entries, 6 to 99996
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Timestamp           16648 non-null  str    
 1   From Bank           16648 non-null  int64  
 2   Account             16648 non-null  str    
 3   To Bank             16648 non-null  int64  
 4   Account.1           16648 non-null  str    
 5   Amount Received     16648 non-null  float64
 6   Receiving Currency  16648 non-null  str    
 7   Amount Paid         16648 non-null  float64
 8   Payment Currency    16648 non-null  str    
 9   Payment Format      16648 non-null  str    
 10  Is Laundering       16648 non-null  int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 1.5 MB
